[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jalonso1979/puremacro/blob/main/curso/notebooks/T00_B_configura_tu_modelo_local_es.ipynb)

> **Nota para Google Colab**: la celda siguiente instala `puremacro` y, si hace falta, descarga del repositorio público [jalonso1979/puremacro](https://github.com/jalonso1979/puremacro) sólo la carpeta `curso/notebooks` (auxiliares, datos congelados y modelos). No pide acceso a Google Drive.

In [1]:
# Configuración del entorno: Google Colab, instalación local o Juno (iPad)
import os
import sys
from pathlib import Path

def _safe_probe(p, is_dir=None):
    """¿Existe la ruta? Sin lanzar OSError/PermissionError en el sandbox de Juno."""
    try:
        p = Path(p)
        if is_dir is True:
            return p.is_dir()
        if is_dir is False:
            return p.is_file()
        return p.exists()
    except (OSError, PermissionError, TypeError, ValueError):
        return False

if "google.colab" in sys.modules:
    !pip install -q "puremacro>=4.8.1,<5"
    if not _safe_probe(Path.cwd() / "_nbstyle.py", is_dir=False):
        # Clon parcial del repositorio público: sólo curso/notebooks, sin historia
        _repo = Path("/content/puremacro")
        if not _safe_probe(_repo, is_dir=True):
            !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/jalonso1979/puremacro.git {_repo}
        !git -C {_repo} sparse-checkout set curso/notebooks 2>&1 | tail -n 2
        os.chdir(_repo / "curso/notebooks")

# Carpeta con los auxiliares del curso (_nbstyle, _datos) en sys.path, sin salir del sandbox
_candidatos = [Path.cwd(), Path.cwd() / "notebooks", Path.cwd() / "curso/notebooks"]
try:
    _candidatos.insert(0, Path(__file__).resolve().parent)  # Juno define __file__
except (NameError, OSError):
    pass
for _c in _candidatos:
    if _safe_probe(_c / "_nbstyle.py", is_dir=False):
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break

# Gráficas en línea y estilo del curso
%matplotlib inline
try:
    import _nbstyle
    _nbstyle.apply_style()
except ImportError:
    pass

# Configura tu modelo local

Esta lección te dice cómo instalar en tu equipo el modelo de lenguaje del curso, Qwen3-4B, con el que los laboratorios leen y puntúan texto, y comprueba que responde. Nada de esto necesita una clave de API ni una cuenta: el modelo corre en tu propio equipo y, después de descargarlo una vez, sin red. Si no logras instalarlo, el modo caché basta para todos los laboratorios: cada uno corre completo con las respuestas de referencia que trae el curso.

In [2]:
import _llm, sys, platform

# Pasos de instalación por plataforma; la celda imprime sólo los de tu equipo.
# El último paso de cada una es la descarga del modelo, con su tamaño.
PASOS = {
    "ios": "\n".join([
        "iPad (beta de Juno)",
        "  1. Instala la beta de Juno con la invitación de TestFlight que te envía el profesor.",
        "  2. Instala puremacro desde el gestor de paquetes de Juno.",
        "  3. Nada más: mlx-lm viene con la beta y descarga el modelo la primera vez que lo usas,",
        "     en las celdas de abajo. Deja Juno en primer plano mientras descarga.",
        "Qwen3-4B, 4 bits: unos 2.5 GB",
    ]),
    "darwin": "\n".join([
        "Mac",
        "  Con Apple silicon (chip M1 o posterior):",
        "  1. Instala mlx-lm en el Python de tus cuadernos: pip install mlx-lm",
        "  2. El modelo se descarga la primera vez que lo usas, en las celdas de abajo.",
        "  Con procesador Intel (o si pip no encuentra mlx-lm):",
        "  1. Instala la app de Ollama (ollama.com) y ábrela.",
        "  2. En una terminal, descarga el modelo: ollama pull qwen3:4b",
        "Qwen3-4B, 4 bits: unos 2.5 GB",
    ]),
    "win32": "\n".join([
        "Windows",
        "  1. Instala Ollama (ollama.com) y ábrelo.",
        "     Si no puedes instalar programas, un llamafile corrido como servidor local también sirve.",
        "  2. En una terminal (PowerShell), descarga el modelo: ollama pull qwen3:4b",
        "Qwen3-4B, 4 bits: unos 2.5 GB",
    ]),
    "linux": "\n".join([
        "Linux",
        "  1. Instala Ollama (ollama.com) y deja corriendo su servidor.",
        "     Si no puedes instalar programas, un llamafile corrido como servidor local también sirve.",
        "  2. En una terminal, descarga el modelo: ollama pull qwen3:4b",
        "Qwen3-4B, 4 bits: unos 2.5 GB",
    ]),
    "colab": "\n".join([
        "Google Colab",
        "  Por ahora, modo caché: cada laboratorio corre completo desde la caché del curso,",
        "  sin instalar nada. Los modelos en vivo en Colab se activarán más adelante.",
        "Qwen3-4B, 4 bits: unos 2.5 GB",
    ]),
}

if "google.colab" in sys.modules:
    plataforma = "colab"
# Un Python anterior a 3.13 en el iPad dice "darwin"; sólo ahí se pregunta platform.machine().
elif sys.platform == "darwin" and platform.machine().startswith(("iPad", "iPhone")):
    plataforma = "ios"
else:
    plataforma = next((p for p in PASOS if sys.platform.startswith(p)), sys.platform)
print(PASOS.get(plataforma, f"Sin pasos para «{plataforma}»: usa el modo caché (al final de esta lección), "
                            "que basta para todos los laboratorios."))

Mac
  Con Apple silicon (chip M1 o posterior):
  1. Instala mlx-lm en el Python de tus cuadernos: pip install mlx-lm
  2. El modelo se descarga la primera vez que lo usas, en las celdas de abajo.
  Con procesador Intel (o si pip no encuentra mlx-lm):
  1. Instala la app de Ollama (ollama.com) y ábrela.
  2. En una terminal, descarga el modelo: ollama pull qwen3:4b
Qwen3-4B, 4 bits: unos 2.5 GB


## La descarga es tarea

Haz la instalación y la descarga en casa, antes de la sesión del laboratorio: el modelo pesa unos 2.5 GB y, si todo el grupo lo descarga a la vez en el salón, la red no alcanza. Con Ollama, la descarga es el `ollama pull` de los pasos de arriba; si no, el modelo lo descarga la primera celda que lo corre. Con el modelo de referencia, `qwen3-4b`, esa es la celda de la comparación en vivo, casi al final de esta lección (la que imprime `referencia:` y `en vivo:`): corre tu modelo aunque la caché del curso ya tenga la respuesta, que la celda de autoverificación sólo lee. Con el modelo pequeño, `qwen3-1.7b`, es la celda de autoverificación, porque la caché del curso no trae respuestas para ese modelo. En el iPad, deja Juno en primer plano y la pantalla encendida mientras corren las dos celdas: si cambias de app o se bloquea la pantalla, iPadOS suspende Juno y la descarga se detiene. Usa una red Wi-Fi estable y sin límite de datos, no los datos del celular; si la descarga se corta, vuelve a correr la celda que estaba descargando (o el `ollama pull`). Cuando termines la instalación, corre la celda siguiente: imprime un informe breve de lo que encuentra el ayudante del curso y le hace una pregunta al modelo.

In [3]:
# Autoverificación: la respuesta de referencia, lo que encuentra el ayudante del curso y una
# pregunta a tu modelo. La referencia se lee primero, de la caché del curso con el modelo de
# referencia (qwen3-4b), antes de que tu modelo corra y pueda guardar una respuesta suya.
Q = "Responde con una palabra: ¿qué mide el PIB?"
ref = _llm.get_llm(mode="cache", model=_llm.REFERENCE_MODEL).try_complete(Q, max_tokens=16)
llm = _llm.get_llm()
print(llm.report())
r = llm.try_complete(Q, max_tokens=16)
print(r if r is not None else "(sin modelo: modo caché)")

Modelo local del curso
  modo:     cache (sólo la caché del curso; sin motores ni red)
  modelo:   qwen3-4b
  caché:    notebooks/data_curso/llm_cache.sqlite (22 filas)
  backend:  fallback (puremacro 4.8.1; sin puremacro.llm)
Valor económico total.


## Respuesta de referencia y respuesta en vivo

La celda de autoverificación leyó la respuesta de referencia antes de que corriera tu modelo: la de Qwen3-4B en 4 bits, calculada una vez por el profesor y guardada en la caché del curso. La celda siguiente le hace la misma pregunta a tu modelo local, sin pasar por la caché, e imprime las dos. Las dos pueden diferir aunque el modelo y la pregunta sean los mismos. La cuantización guarda cada peso en 4 bits y cada formato (MLX, GGUF) redondea a su manera; el motor (MLX, llama.cpp u Ollama) suma en otro orden y con otros núcleos de cálculo. Esas diferencias mueven los últimos decimales de las probabilidades del modelo, y cuando dos palabras casi empatan basta eso para que la respuesta cambie. Por eso, en los trabajos que se califican reporta las dos: la de referencia, que cualquiera reproduce, y la tuya en vivo, con el motor y el modelo que usaste. La referencia del profesor existe sólo para las preguntas a `qwen3-4b` que trae la caché del curso: si la línea de referencia dice `None`, la caché del curso no tiene esta; pregúntale al profesor. Mientras la caché del curso no la tenga, la línea de referencia de una corrida posterior tampoco es la del profesor: si una corrida de la celda de autoverificación con `qwen3-4b` obtuvo una respuesta en modo auto, guardó esa respuesta en el archivo de la caché del curso, y las líneas de referencia posteriores la imprimen. Un `None` en la línea en vivo quiere decir que no hubo respuesta en vivo; la nota `[modelo local]` que aparece antes dice por qué.

In [4]:
# Tu modelo local con la misma pregunta Q, junto a la referencia `ref` que leyó la celda de
# autoverificación. En modo auto, "off" corre el modelo aunque la caché tenga la respuesta; en
# modo cache no hay respuesta en vivo y la línea "en vivo" también sale de la caché.
modo_vivo = "off" if llm.mode == "auto" else llm.mode
vivo = _llm.get_llm(mode=modo_vivo).try_complete(Q, max_tokens=16)
print("referencia:", ref)
print("en vivo:", vivo)

referencia: Valor económico total.
en vivo: Valor económico total.


## Si nada funciona

No pasa nada: todos los laboratorios corren completos desde la caché del curso (`data_curso/llm_cache.sqlite`, que viaja con los cuadernos), con modelo o sin él. La caché guarda las respuestas de `qwen3-4b` del profesor a las preguntas que trae el curso; si falta alguna, el laboratorio imprime `None` y una nota en lugar de detenerse: avísale al profesor. Para no buscar motores en absoluto, corre `import os; os.environ["MAV_LLM_MODE"] = "cache"` en una celda antes de usar el modelo. El modelo pequeño es sólo un respaldo para equipos con menos de 5 GB de memoria para el modelo (en modo auto, el informe de la autoverificación avisa entonces que `qwen3-4b` no cabe): ahí, usa `os.environ["PUREMACRO_LLM_MODEL"] = "qwen3-1.7b"`, sabiendo que la caché del curso no trae respuestas para él y que las suyas se alejan más de la referencia, o quédate en modo caché. Para pedir ayuda, copia el informe de la autoverificación (no lleva rutas absolutas) y pregunta en Canvas, el canal oficial del curso, o en una asesoría.